In [6]:
# Cell 1
import pandas as pd
import os

# Optional: Set pandas to show all columns so nothing gets hidden
pd.set_option('display.max_columns', None)

In [7]:
# Cell 2
# Target: FEMA Region 5 (Illinois/Midwest) 
TARGET_STATES = ['IL', 'IN', 'MI', 'MN', 'OH', 'WI']

# Define relative paths for the Keeling workspace
RAW_DATA_DIR = './data/raw'
PROCESSED_DATA_DIR = './data/processed'

# Ensure the directories exist
os.makedirs(RAW_DATA_DIR, exist_ok=True)
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)

print("Directories verified. Ready to load data.")

Directories verified. Ready to load data.


In [8]:
# Cell 3
def load_and_inspect_eagle_i(filename):
    """
    Loads raw EAGLE-I data, filters for the pilot region, 
    and displays the initial dataframe structure.
    """
    file_path = os.path.join(RAW_DATA_DIR, filename)
    print(f"Attempting to load: {file_path}")
    
    try:
        df = pd.read_csv(file_path)
        
        # Standardize column names to lowercase
        df.columns = [col.lower() for col in df.columns]
        
        # Filter down to FEMA Region 5
        if 'state' in df.columns:
            df_filtered = df[df['state'].isin(TARGET_STATES)].copy()
        elif 'state_code' in df.columns:
            df_filtered = df[df['state_code'].isin(TARGET_STATES)].copy()
        else:
            print("Warning: Could not find a state column. Available columns are:")
            print(df.columns)
            return df
            
        print(f"Success! Filtered from {len(df)} total rows down to {len(df_filtered)} Region 5 rows.")
        return df_filtered
        
    except FileNotFoundError:
        print(f"Error: The file {filename} was not found in the {RAW_DATA_DIR} directory.")
        return None

In [9]:
# Cell 4
# Replace 'your_downloaded_file.csv' with the actual name of the file
# from the DOE EAGLE-I dataset 
eagle_df = load_and_inspect_eagle_i("your_downloaded_file.csv")

# Display the first 5 rows to verify the structure
if eagle_df is not None:
    display(eagle_df.head())

Attempting to load: ./data/raw/your_downloaded_file.csv
Error: The file your_downloaded_file.csv was not found in the ./data/raw directory.


In [11]:
# Cell 5: Setup Zarr and Icechunk for GEFSv12
import os
import xarray as xr
import zarr
from icechunk import Repository, local_filesystem_storage

# Define where the Icechunk repository will live on Keeling
ICECHUNK_REPO_DIR = './data/processed/gefs_icechunk_repo'

# Initialize a new Icechunk repository (Run this only once to create it)
if not os.path.exists(ICECHUNK_REPO_DIR):
    storage = local_filesystem_storage(ICECHUNK_REPO_DIR)
    repo = Repository.create(storage)
    print(f"Initialized new Icechunk repository at {ICECHUNK_REPO_DIR}")
else:
    # Open existing repo
    storage = local_filesystem_storage(ICECHUNK_REPO_DIR)
    repo = Repository.open(storage)
    print("Opened existing Icechunk repository.")

# Create a working session on the main branch
session = repo.writable_session("main")

# --- Example of how we will process the GRIB2 files ---
def process_grib_to_zarr(grib_file_path, session, is_first_file=False):
    """
    Opens a downloaded GEFSv12 GRIB2 file and writes it to the Icechunk/Zarr store.
    """
    print(f"Processing {grib_file_path}...")
    
    # Open the GRIB2 file using xarray and the cfgrib/pygrib engine
    # We will filter for specific ingredients like shear, CAPE, moisture later
    ds = xr.open_dataset(grib_file_path, engine='cfgrib')
    
    # Get the Zarr store interface from our Icechunk session
    store = session.store
    
    # Write to the Zarr store. 
    if is_first_file:
        # Create the store and write the initial data
        ds.to_zarr(store, group="gefsv12_pilot", mode='w')
    else:
        # Append to the existing store along the time dimension
        ds.to_zarr(store, group="gefsv12_pilot", mode='a', append_dim='time')
    
    # Commit the changes to the Icechunk repo
    commit_id = session.commit(f"Added GEFSv12 data from {grib_file_path}")
    print(f"Successfully committed to Icechunk store. Commit ID: {commit_id}")
    
    return commit_id

  2026-03-03T04:40:01.767938Z  WARN icechunk::storage::object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk/src/storage/object_store.rs:81

Opened existing Icechunk repository.
